# Physiological artifact cleaning for the CSP neural network

This notebook adds four preprocessing steps to the artifact-retained motor-imagery trials:

1. Common-average reference (CAR) across the 27 scalp electrodes.
2. ICA removal of components strongly correlated with the three EOG channels.
3. Linear EOG regression to remove residual ocular contribution.
4. Robust EMG-informed rejection using `EMGg` and `EMGd`.

ICA is applied before EOG regression because regression would remove the linear EOG correlation used to identify ocular ICA components. This ordering implements both ocular methods without making ICA a mathematical no-op.

Only training and validation participants are processed. Test participants remain untouched.

## 1. Imports, paths, and fixed cleaning rules

All thresholds are fixed before processing:

- At most three ICA components may be removed.
- A component must have absolute EOG correlation of at least 0.30.
- An EMG trial is rejected when either channel's log-power robust z-score exceeds 4.
- The final EEG is filtered into mu (8–13 Hz) and beta (13–30 Hz), then represented as normalized trial covariance matrices for CSP.

In [1]:
from pathlib import Path
import time
import warnings

import mne
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import signal
from sklearn.covariance import ledoit_wolf
from sklearn.decomposition import FastICA
from sklearn.exceptions import ConvergenceWarning

mne.set_log_level('ERROR')
SEED = 42

def find_project_data(start=Path.cwd()):
    for parent in (start.resolve(), *start.resolve().parents):
        candidate = parent / 'data' / 'processed' / 'Signals'
        if candidate.is_dir():
            return parent / 'data'
    raise FileNotFoundError('Could not locate data/processed/Signals.')

DATA_ROOT = find_project_data()
SIGNALS_ROOT = DATA_ROOT / 'processed' / 'Signals'
INDEX_PATH = DATA_ROOT / 'processed' / 'modeling_index' / 'trial_modeling_index_with_split.csv'
RETAINED_PATH = (
    DATA_ROOT / 'processed' / 'all_trials_time_frequency_artifact_rejected'
    / 'retained_trials.csv'
)
OUTPUT_ROOT = DATA_ROOT / 'processed' / 'physiological_artifact_cleaning'
COVARIANCE_ROOT = OUTPUT_ROOT / 'covariances_by_file'
QUALITY_ROOT = OUTPUT_ROOT / 'quality_by_file'
DEVELOPMENT_INDEX_PATH = OUTPUT_ROOT / 'development_index_physio_clean.csv'
MANIFEST_PATH = OUTPUT_ROOT / 'physiological_artifact_manifest.csv'
FILE_SUMMARY_PATH = OUTPUT_ROOT / 'file_summary.csv'
for directory in (COVARIANCE_ROOT, QUALITY_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

EEG_CHANNELS = [
    'Fz', 'FCz', 'Cz', 'CPz', 'Pz',
    'C1', 'C3', 'C5', 'C2', 'C4', 'C6',
    'F4', 'FC2', 'FC4', 'FC6', 'CP2', 'CP4', 'CP6', 'P4',
    'F3', 'FC1', 'FC3', 'FC5', 'CP1', 'CP3', 'CP5', 'P3',
]
EOG_CHANNELS = ['EOG1', 'EOG2', 'EOG3']
EMG_CHANNELS = ['EMGg', 'EMGd']
BANDS_HZ = {'mu': (8.0, 13.0), 'beta': (13.0, 30.0)}
STIMULUS_SECONDS = (0.5, 5.0)
CANDIDATE_WINDOWS = {
    '0.5_to_3.0': (0.5, 3.0),
    '0.5_to_4.0': (0.5, 4.0),
    '1.0_to_4.0': (1.0, 4.0),
    '1.0_to_5.0': (1.0, 5.0),
    '0.5_to_5.0': STIMULUS_SECONDS,
}
BAND_TUNING_HZ = {
    'mu_7_12': (7.0, 12.0),
    'mu_8_12': (8.0, 12.0),
    'mu_8_13': (8.0, 13.0),
    'mu_8_14': (8.0, 14.0),
    'beta_13_25': (13.0, 25.0),
    'beta_13_30': (13.0, 30.0),
    'beta_15_25': (15.0, 25.0),
    'beta_15_30': (15.0, 30.0),
}
BAND_TUNING_WINDOW = (0.5, 3.0)
ICA_CORRELATION_THRESHOLD = 0.30
MAX_ICA_COMPONENTS = 3
ICA_COMPONENTS = 26  # CAR reduces 27-channel rank by one.
ICA_FIT_SAMPLES = 5_000
ICA_MAX_ITER = 500
ICA_TOL = 5e-3
ICA_ALGORITHM = 'deflation'
EMG_ROBUST_Z_THRESHOLD = 4.0

print('Output:', OUTPUT_ROOT)

Output: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/physiological_artifact_cleaning


## 2. Select development trials and verify test isolation

The existing modeling index contains trials that passed the earlier extreme-amplitude and flat-channel checks. This notebook adds physiological cleaning to those trials without changing the participant split.

In [2]:
model_index = pd.read_csv(INDEX_PATH)
development = model_index.loc[
    model_index['split'].isin(['train', 'validation'])
].copy()
retained = pd.read_csv(
    RETAINED_PATH, usecols=['source_file', 'trial', 'cue_sample']
)
development = development.merge(
    retained, on=['source_file', 'trial'], how='left', validate='one_to_one'
).reset_index(drop=True)

assert development['cue_sample'].notna().all()
assert set(development['split']) == {'train', 'validation'}
assert 'test' not in set(development['split'])
assert development['sample_id'].is_unique
assert development.groupby('split')['participant'].nunique().to_dict() == {
    'train': 55, 'validation': 12,
}

display(
    development.groupby('split').agg(
        participants=('participant', 'nunique'),
        recordings=('source_file', 'nunique'),
        trials=('sample_id', 'size'),
    ).reset_index()
)
print('Test rows loaded: 0')

,split,participants,recordings,trials
0,train,55,320,12156
1,validation,12,69,2570


Test rows loaded: 0


## 3. Cleaning and covariance functions

The continuous EEG is first filtered to 1–40 Hz for stable ICA. EOG-correlated ICA sources are removed, followed by residual EOG regression.

EMG activity is measured within each stimulus epoch after a 30–100 Hz filter. Thresholds are calculated independently within each recording using the median and median absolute deviation (MAD), which adapts to participant and sensor scale without using class labels.

In [3]:
def derivative_paths(source_file):
    relative = Path(source_file).with_suffix('')
    covariance = (
        COVARIANCE_ROOT / relative.parent
        / f'{relative.name}_physio_clean_covariances.npz'
    )
    quality = (
        QUALITY_ROOT / relative.parent
        / f'{relative.name}_physio_quality.csv'
    )
    return covariance, quality

def outputs_valid(source_file, expected_ids):
    covariance_path, quality_path = derivative_paths(source_file)
    if not covariance_path.is_file() or not quality_path.is_file():
        return False
    try:
        quality = pd.read_csv(quality_path)
        with np.load(covariance_path, allow_pickle=False) as saved:
            return (
                len(quality) == len(expected_ids)
                and np.array_equal(quality['sample_id'].astype(str), expected_ids)
                and saved['covariances'].shape == (
                    int(quality['retained_after_physio_cleaning'].sum()),
                    2, 27, 27,
                )
                and saved['window_covariances'].shape == (
                    int(quality['retained_after_physio_cleaning'].sum()),
                    len(CANDIDATE_WINDOWS), 2, 27, 27,
                )
                and saved['window_ledoit_wolf_covariances'].shape == (
                    int(quality['retained_after_physio_cleaning'].sum()),
                    len(CANDIDATE_WINDOWS), 2, 27, 27,
                )
                and saved['band_tuning_ledoit_wolf_covariances'].shape == (
                    int(quality['retained_after_physio_cleaning'].sum()),
                    len(BAND_TUNING_HZ), 27, 27,
                )
                and np.array_equal(
                    saved['band_tuning_names'].astype(str),
                    np.asarray(list(BAND_TUNING_HZ)),
                )
                and np.array_equal(
                    saved['window_names'].astype(str),
                    np.asarray(list(CANDIDATE_WINDOWS)),
                )
                and np.isfinite(saved['covariances']).all()
                and np.isfinite(saved['window_covariances']).all()
                and np.isfinite(saved['window_ledoit_wolf_covariances']).all()
                and np.isfinite(
                    saved['band_tuning_ledoit_wolf_covariances']
                ).all()
                and float(saved['emg_robust_z_threshold']) == EMG_ROBUST_Z_THRESHOLD
                and float(saved['ica_correlation_threshold']) == ICA_CORRELATION_THRESHOLD
                and int(saved['ica_fit_samples']) == ICA_FIT_SAMPLES
                and int(saved['ica_max_iter']) == ICA_MAX_ITER
                and str(saved['ica_algorithm']) == ICA_ALGORITHM
            )
    except Exception:
        return False

def robust_z(values):
    values = np.asarray(values, dtype=float)
    median = np.median(values)
    mad = np.median(np.abs(values - median))
    scale = 1.4826 * mad
    if not np.isfinite(scale) or scale <= 1e-12:
        scale = np.std(values)
    if not np.isfinite(scale) or scale <= 1e-12:
        return np.zeros_like(values)
    return (values - median) / scale

def normalized_covariance(epoch):
    epoch = epoch - epoch.mean(axis=1, keepdims=True)
    covariance = epoch @ epoch.T
    trace = np.trace(covariance)
    if not np.isfinite(trace) or trace <= 0:
        raise ValueError('Invalid covariance trace.')
    return covariance / trace

def remove_ica_eog(eeg_car, eog, sfreq):
    analysis_sos = signal.butter(
        4, [1.0, 40.0], btype='bandpass', fs=sfreq, output='sos'
    )
    eeg_1_40 = signal.sosfiltfilt(analysis_sos, eeg_car, axis=1)
    eog_1_15 = signal.sosfiltfilt(
        signal.butter(4, [1.0, 15.0], btype='bandpass', fs=sfreq, output='sos'),
        eog, axis=1,
    )

    decimation = max(1, int(round(sfreq / 128.0)))
    sample_positions = np.arange(0, eeg_1_40.shape[1], decimation)
    if len(sample_positions) > ICA_FIT_SAMPLES:
        sample_positions = sample_positions[
            np.linspace(0, len(sample_positions) - 1, ICA_FIT_SAMPLES, dtype=int)
        ]
    fit_data = eeg_1_40[:, sample_positions].T

    ica = FastICA(
        n_components=ICA_COMPONENTS,
        whiten='unit-variance',
        random_state=SEED,
        max_iter=ICA_MAX_ITER,
        tol=ICA_TOL,
        algorithm=ICA_ALGORITHM,
    )
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        ica.fit(fit_data)
    converged = not any(issubclass(item.category, ConvergenceWarning) for item in caught)

    correlation_positions = np.arange(0, eeg_1_40.shape[1], decimation)
    sources = ica.transform(eeg_1_40[:, correlation_positions].T)
    eog_decimated = eog_1_15[:, correlation_positions].T
    correlations = np.corrcoef(
        sources.T, eog_decimated.T
    )[:ICA_COMPONENTS, ICA_COMPONENTS:]
    component_scores = np.max(np.abs(correlations), axis=1)
    eligible = np.flatnonzero(component_scores >= ICA_CORRELATION_THRESHOLD)
    removed = eligible[np.argsort(component_scores[eligible])[::-1]][:MAX_ICA_COMPONENTS]

    full_sources = ica.transform(eeg_1_40.T)
    if len(removed):
        full_sources[:, removed] = 0.0
    reconstructed = ica.inverse_transform(full_sources).T
    return reconstructed, removed, component_scores, converged, int(ica.n_iter_)

def regress_residual_eog(eeg, eog):
    eog_centered = eog - eog.mean(axis=1, keepdims=True)
    eeg_centered = eeg - eeg.mean(axis=1, keepdims=True)
    gram = eog_centered @ eog_centered.T
    ridge = max(np.trace(gram) / len(gram) * 1e-6, np.finfo(float).eps)
    coefficients = (
        eeg_centered @ eog_centered.T
        @ np.linalg.inv(gram + ridge * np.eye(len(gram)))
    )
    return eeg_centered - coefficients @ eog_centered

print('Cleaning functions ready.')

Cleaning functions ready.


## 4. Apply cleaning to every development recording

Each recording is checkpointed independently. Rerunning this cell skips files whose covariance and quality outputs match the fixed configuration.

In [4]:
def process_recording(source_file, rows):
    rows = rows.sort_values('sample_id').reset_index(drop=True)
    expected_ids = rows['sample_id'].to_numpy(dtype=str)
    covariance_path, quality_path = derivative_paths(source_file)
    if outputs_valid(source_file, expected_ids):
        return {'source_file': source_file, 'status': 'cached'}

    raw = mne.io.read_raw_gdf(
        SIGNALS_ROOT / source_file, preload=True, verbose='ERROR'
    )
    try:
        sfreq = float(raw.info['sfreq'])
        assert sfreq == 512.0
        eeg = raw.get_data(picks=EEG_CHANNELS)
        eog = raw.get_data(picks=EOG_CHANNELS)
        emg = raw.get_data(picks=EMG_CHANNELS)
    finally:
        raw.close()

    # Step 1: common-average reference.
    eeg_car = eeg - eeg.mean(axis=0, keepdims=True)
    del eeg

    # Steps 2 and 3: ICA ocular removal, then residual EOG regression.
    eeg_ica, removed, component_scores, converged, iterations = remove_ica_eog(
        eeg_car, eog, sfreq
    )
    del eeg_car
    eeg_clean = regress_residual_eog(eeg_ica, eog)
    del eeg_ica

    filtered_eeg = []
    for low_hz, high_hz in BANDS_HZ.values():
        sos = signal.butter(
            4, [low_hz, high_hz], btype='bandpass', fs=sfreq, output='sos'
        )
        filtered_eeg.append(signal.sosfiltfilt(sos, eeg_clean, axis=1))

    # Step 4: EMG activity in 30–100 Hz.
    emg_filtered = signal.sosfiltfilt(
        signal.butter(
            4, [30.0, 100.0], btype='bandpass', fs=sfreq, output='sos'
        ),
        emg, axis=1,
    )
    del emg

    start_offset = int(round(STIMULUS_SECONDS[0] * sfreq))
    stop_offset = int(round(STIMULUS_SECONDS[1] * sfreq))
    trial_window_covariances = np.empty(
        (len(rows), len(CANDIDATE_WINDOWS), 2, 27, 27), dtype=np.float32
    )
    trial_window_ledoit_wolf_covariances = np.empty_like(
        trial_window_covariances
    )
    trial_band_tuning_ledoit_wolf_covariances = np.empty(
        (len(rows), len(BAND_TUNING_HZ), 27, 27), dtype=np.float32
    )
    log_emg_power = np.empty((len(rows), 2), dtype=float)

    for row_number, row in enumerate(rows.itertuples(index=False)):
        start = int(row.cue_sample) + start_offset
        stop = int(row.cue_sample) + stop_offset
        if start < 0 or stop > filtered_eeg[0].shape[1]:
            raise ValueError(f'Invalid trial bounds for {row.sample_id}.')
        for window_number, (window_start, window_stop) in enumerate(
            CANDIDATE_WINDOWS.values()
        ):
            covariance_start = int(row.cue_sample) + int(
                round(window_start * sfreq)
            )
            covariance_stop = int(row.cue_sample) + int(
                round(window_stop * sfreq)
            )
            if covariance_start < 0 or covariance_stop > filtered_eeg[0].shape[1]:
                raise ValueError(f'Invalid trial bounds for {row.sample_id}.')
            for band_number, band_signal in enumerate(filtered_eeg):
                epoch = band_signal[:, covariance_start:covariance_stop]
                trial_window_covariances[
                    row_number, window_number, band_number
                ] = normalized_covariance(
                    epoch
                )
                regularized_covariance, _ = ledoit_wolf(
                    epoch.T, assume_centered=False
                )
                trial_window_ledoit_wolf_covariances[
                    row_number, window_number, band_number
                ] = regularized_covariance / np.trace(regularized_covariance)
        emg_epoch = emg_filtered[:, start:stop]
        log_emg_power[row_number] = np.log(
            np.maximum(np.mean(np.square(emg_epoch), axis=1), np.finfo(float).tiny)
        )

    del filtered_eeg
    tuning_start_offset = int(round(BAND_TUNING_WINDOW[0] * sfreq))
    tuning_stop_offset = int(round(BAND_TUNING_WINDOW[1] * sfreq))
    for band_number, limits in enumerate(BAND_TUNING_HZ.values()):
        tuning_sos = signal.butter(
            4, limits, btype='bandpass', fs=sfreq, output='sos'
        )
        tuning_signal = signal.sosfiltfilt(tuning_sos, eeg_clean, axis=1)
        for row_number, row in enumerate(rows.itertuples(index=False)):
            start = int(row.cue_sample) + tuning_start_offset
            stop = int(row.cue_sample) + tuning_stop_offset
            if start < 0 or stop > tuning_signal.shape[1]:
                raise ValueError(f'Invalid tuning bounds for {row.sample_id}.')
            covariance, _ = ledoit_wolf(
                tuning_signal[:, start:stop].T, assume_centered=False
            )
            trial_band_tuning_ledoit_wolf_covariances[
                row_number, band_number
            ] = covariance / np.trace(covariance)
        del tuning_signal
    del eeg_clean

    emg_z = np.column_stack([
        robust_z(log_emg_power[:, channel]) for channel in range(2)
    ])
    emg_rejected = np.any(emg_z > EMG_ROBUST_Z_THRESHOLD, axis=1)
    retained_mask = ~emg_rejected

    quality = rows[[
        'sample_id', 'dataset', 'participant', 'run', 'phase',
        'source_file', 'trial', 'class_label', 'label_id', 'split',
    ]].copy()
    quality['emg_g_log_power'] = log_emg_power[:, 0]
    quality['emg_d_log_power'] = log_emg_power[:, 1]
    quality['emg_g_robust_z'] = emg_z[:, 0]
    quality['emg_d_robust_z'] = emg_z[:, 1]
    quality['emg_rejected'] = emg_rejected
    quality['retained_after_physio_cleaning'] = retained_mask
    quality['ica_components_removed'] = len(removed)
    quality['ica_removed_indices'] = '|'.join(map(str, removed.tolist()))
    quality['ica_max_eog_correlation'] = float(component_scores.max())
    quality['ica_converged'] = converged
    quality['ica_iterations'] = iterations

    covariance_path.parent.mkdir(parents=True, exist_ok=True)
    quality_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_covariance = covariance_path.with_name(
        covariance_path.name + '.tmp.npz'
    )
    np.savez_compressed(
        temporary_covariance,
        covariances=trial_window_covariances[
            retained_mask, list(CANDIDATE_WINDOWS).index('0.5_to_5.0')
        ],
        window_covariances=trial_window_covariances[retained_mask],
        window_ledoit_wolf_covariances=(
            trial_window_ledoit_wolf_covariances[retained_mask]
        ),
        covariance_estimator=np.asarray('Ledoit-Wolf'),
        band_tuning_ledoit_wolf_covariances=(
            trial_band_tuning_ledoit_wolf_covariances[retained_mask]
        ),
        band_tuning_names=np.asarray(list(BAND_TUNING_HZ)),
        band_tuning_hz=np.asarray(list(BAND_TUNING_HZ.values())),
        band_tuning_window=np.asarray(BAND_TUNING_WINDOW),
        window_names=np.asarray(list(CANDIDATE_WINDOWS)),
        window_seconds=np.asarray(list(CANDIDATE_WINDOWS.values())),
        sample_ids=expected_ids[retained_mask],
        labels=rows['label_id'].to_numpy(dtype=np.int64)[retained_mask],
        electrodes=np.asarray(EEG_CHANNELS),
        bands=np.asarray(list(BANDS_HZ)),
        stimulus_seconds=np.asarray(STIMULUS_SECONDS),
        ica_correlation_threshold=np.asarray(ICA_CORRELATION_THRESHOLD),
        max_ica_components=np.asarray(MAX_ICA_COMPONENTS),
        ica_fit_samples=np.asarray(ICA_FIT_SAMPLES),
        ica_max_iter=np.asarray(ICA_MAX_ITER),
        ica_algorithm=np.asarray(ICA_ALGORITHM),
        emg_robust_z_threshold=np.asarray(EMG_ROBUST_Z_THRESHOLD),
        cleaning_method=np.asarray(
            'CAR -> ICA EOG -> residual EOG regression -> EMG rejection'
        ),
    )
    temporary_covariance.replace(covariance_path)
    temporary_quality = quality_path.with_name(quality_path.name + '.tmp.csv')
    quality.to_csv(temporary_quality, index=False)
    temporary_quality.replace(quality_path)
    return {
        'source_file': source_file,
        'status': 'processed',
        'trials': len(rows),
        'retained': int(retained_mask.sum()),
        'emg_rejected': int(emg_rejected.sum()),
        'ica_components_removed': len(removed),
        'ica_converged': converged,
    }

file_groups = list(development.groupby('source_file', sort=True))
processing_rows = []
started = time.perf_counter()
for file_number, (source_file, rows) in enumerate(file_groups, start=1):
    result = process_recording(source_file, rows)
    processing_rows.append(result)
    if file_number == 1 or file_number % 10 == 0 or file_number == len(file_groups):
        print(
            f'[{file_number:>3}/{len(file_groups)}] {result["status"]:<9} '
            f'{source_file} ({(time.perf_counter() - started) / 60:.1f} min)'
        )

processing_log = pd.DataFrame(processing_rows)
processing_log.to_csv(OUTPUT_ROOT / 'processing_log.csv', index=False)
display(processing_log['status'].value_counts().rename_axis('status').to_frame())

[  1/389] cached    DATA A/A1/A1_R2_acquisition.gdf (0.0 min)
[ 10/389] cached    DATA A/A12/A12_R5_onlineT.gdf (0.0 min)
[ 20/389] cached    DATA A/A14/A14_R3_onlineT.gdf (0.0 min)
[ 30/389] cached    DATA A/A16/A16_R1_acquisition.gdf (0.0 min)
[ 40/389] cached    DATA A/A19/A19_R5_onlineT.gdf (0.0 min)
[ 50/389] cached    DATA A/A20/A20_R3_onlineT.gdf (0.0 min)
[ 60/389] cached    DATA A/A24/A24_R2_acquisition.gdf (0.0 min)
[ 70/389] cached    DATA A/A25/A25_R6_onlineT.gdf (0.0 min)
[ 80/389] cached    DATA A/A27/A27_R4_onlineT.gdf (0.0 min)
[ 90/389] cached    DATA A/A3/A3_R2_acquisition.gdf (0.0 min)
[100/389] cached    DATA A/A30/A30_R6_onlineT.gdf (0.0 min)
[110/389] cached    DATA A/A32/A32_R4_onlineT.gdf (0.0 min)
[120/389] cached    DATA A/A34/A34_R2_acquisition.gdf (0.0 min)
[130/389] cached    DATA A/A35/A35_R6_onlineT.gdf (0.0 min)
[140/389] cached    DATA A/A37/A37_R4_onlineT.gdf (0.0 min)
[150/389] cached    DATA A/A39/A39_R2_acquisition.gdf (0.0 min)
[160/389] cached    

,count
status,
cached,389


## 5. Consolidate the cleaned index and audit retention

The new modeling index contains only trials retained after EMG-informed rejection and points to the physiological-cleaning covariance file and row for each trial.

In [5]:
quality_frames = []
index_frames = []
for source_file, rows in file_groups:
    covariance_path, quality_path = derivative_paths(source_file)
    expected_ids = rows.sort_values('sample_id')['sample_id'].to_numpy(dtype=str)
    if not outputs_valid(source_file, expected_ids):
        raise RuntimeError(f'Missing or invalid output for {source_file}.')
    quality = pd.read_csv(quality_path)
    quality_frames.append(quality)
    retained_quality = quality.loc[
        quality['retained_after_physio_cleaning']
    ].reset_index(drop=True)
    retained_quality['physio_covariance_relative_path'] = (
        covariance_path.relative_to(DATA_ROOT).as_posix()
    )
    retained_quality['physio_covariance_row'] = np.arange(len(retained_quality))
    index_frames.append(retained_quality)

manifest = pd.concat(quality_frames, ignore_index=True)
clean_index_keys = pd.concat(index_frames, ignore_index=True)[[
    'sample_id', 'physio_covariance_relative_path', 'physio_covariance_row'
]]
clean_index = development.merge(
    clean_index_keys, on='sample_id', how='inner', validate='one_to_one'
)

manifest.to_csv(MANIFEST_PATH, index=False)
clean_index.to_csv(DEVELOPMENT_INDEX_PATH, index=False)

file_summary = (
    manifest.groupby(['source_file', 'dataset', 'participant', 'split'])
    .agg(
        trials=('sample_id', 'size'),
        retained=('retained_after_physio_cleaning', 'sum'),
        emg_rejected=('emg_rejected', 'sum'),
        ica_components_removed=('ica_components_removed', 'first'),
        ica_max_eog_correlation=('ica_max_eog_correlation', 'first'),
        ica_converged=('ica_converged', 'first'),
    )
    .reset_index()
)
file_summary.to_csv(FILE_SUMMARY_PATH, index=False)

assert clean_index['sample_id'].is_unique
assert set(clean_index['split']) == {'train', 'validation'}
assert clean_index.groupby('split')['participant'].nunique().to_dict() == {
    'train': 55, 'validation': 12,
}
assert 'test' not in set(clean_index['split'])

retention_summary = (
    manifest.groupby('split')
    .agg(
        participants=('participant', 'nunique'),
        input_trials=('sample_id', 'size'),
        retained=('retained_after_physio_cleaning', 'sum'),
        emg_rejected=('emg_rejected', 'sum'),
    )
    .reset_index()
)
retention_summary['retention_rate'] = (
    retention_summary['retained'] / retention_summary['input_trials']
)
display(retention_summary)

ica_summary = pd.DataFrame({
    'recordings': [len(file_summary)],
    'ICA_converged': [int(file_summary['ica_converged'].sum())],
    'recordings_with_components_removed': [
        int((file_summary['ica_components_removed'] > 0).sum())
    ],
    'mean_components_removed': [file_summary['ica_components_removed'].mean()],
    'maximum_components_removed': [file_summary['ica_components_removed'].max()],
})
display(ica_summary)

print('Clean development index:', DEVELOPMENT_INDEX_PATH)
print('Test participants processed: NO')

,split,participants,input_trials,retained,emg_rejected,retention_rate
0,train,55,12156,12029,127,0.989552
1,validation,12,2570,2535,35,0.986381


,recordings,ICA_converged,recordings_with_components_removed,mean_components_removed,maximum_components_removed
0,389,389,389,1.825193,3


Clean development index: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/physiological_artifact_cleaning/development_index_physio_clean.csv
Test participants processed: NO


## Handoff to CSP

`development_index_physio_clean.csv` and the covariance files under `covariances_by_file` are the only EEG inputs required by the updated CSP neural-network notebook.

The cleaning algorithm uses no class labels. The participant split is preserved, and the test set remains unprocessed.